# Jupiter flux review — phase2 QA Zarr

Interactive summary of Stokes I flux toward **Jupiter** in pipeline QA Zarr stores
(`pipelineQA-phase2-I-NoTaper-Robust-0-*.zarr`).

For the selected observation day:

1. **Ephemeris** — Astropy `get_body("jupiter", …)` at the **first** dataset time step.
2. **Dynamic spectrum** — flux vs LST × frequency at that fixed RA/Dec. Methods:
   `dynamic_spectrum` (tracked centre pixel), `patch_fit` (shifted 2D Gaussian),
   or `patch_max` (maximum in the tracked patch). Click a cell for patch-fit
   diagnostics when using `patch_fit`.
3. **Sky view** — click a cell in the dynamic spectrum to show that time/frequency
   slice in `astrowidget.SkyWidget`, centered on Jupiter at observation start.

Paths and Zarr naming follow `pipeline_qa_check_phase2.ipynb`.

Flux extraction uses Dask-backed batched I/O and (for `patch_fit`) multiprocess
Gaussian fitting — see **Parallelism notes** at the bottom.

Launch with: `pixi run jupyter lab`

**Run cells in order** (config → setup → optional Dask → launch UI).


In [1]:
from dataclasses import replace
from pathlib import Path

from ovro_lwa_portal.viz.pipeline_qa import PipelineQAConfig

# Edit before running if your staging paths differ.
ZARR_ROOT = Path("/fast/claw")
I_QA_ZARR_STEM = "pipelineQA-phase2-I-NoTaper-Robust-0"

QA_CONFIG = replace(
    PipelineQAConfig.phase2_default(),
    zarr_root=ZARR_ROOT,
    i_qa_zarr_stem=I_QA_ZARR_STEM,
)

# Flux extraction: "dynamic_spectrum" (tracked pixel) or "patch_fit" (Gaussian peak).
FLUX_METHOD = "patch_fit"
# Patch half-width = ceil(scale * max beam FWHM in pixels) at each time/frequency.
PATCH_FIT_SCALE = 3.0
PATCH_FIT_MAX_REDUCED_CHI_SQUARED = 200.0
ZARR_LM_CHUNK = 512

# Optional distributed Dask (see the Dask cell below).
USE_DASK_CLIENT = False
DASK_WORKERS = 6
DASK_THREADS_PER_WORKER = 1
DASK_MEMORY_LIMIT = "16GiB"
DASK_PROCESSES = True  # True for CPU-bound patch_fit; False for I/O-only debugging


In [2]:
from ovro_lwa_portal.viz.jupiter_flux_review_app import (
    JupiterFluxReview,
    JupiterFluxReviewConfig,
    configure_jupiter_flux_review_notebook,
)

configure_jupiter_flux_review_notebook()


In [3]:
if USE_DASK_CLIENT:
    from dask.distributed import Client, get_client

    try:
        dask_client = get_client()
    except ValueError:
        dask_client = Client(
            n_workers=DASK_WORKERS,
            threads_per_worker=DASK_THREADS_PER_WORKER,
            processes=DASK_PROCESSES,
            memory_limit=DASK_MEMORY_LIMIT,
        )
    print(dask_client)
    print(f"Dashboard: {dask_client.dashboard_link}")
else:
    print(
        "Dask Client disabled — radport uses threaded Zarr I/O and a local "
        "process pool for patch_fit / patch_max reductions."
    )

Dask Client disabled — radport uses threaded Zarr I/O and a local process pool for patch_fit / patch_max reductions.


In [4]:
review = JupiterFluxReview(
    QA_CONFIG,
    flux_method=FLUX_METHOD,
    patch_fit_scale=PATCH_FIT_SCALE,
    patch_fit_max_reduced_chi_squared=PATCH_FIT_MAX_REDUCED_CHI_SQUARED,
    review_config=JupiterFluxReviewConfig(zarr_lm_chunk=ZARR_LM_CHUNK),
)
review.panel


Column(max_width=1048, sizing_mode='stretch_width')
    [0] Row(margin=(0, 0, 8, 0), sizing_mode='stretch_width')
        [0] Select(description='Phase2 QA Zarr store.', name='Phase2 QA Zarr (..., options=OrderedDict({'2024-12-18 (...]), value='2024-12-18 (pipelineQA-ph..., width=520)
        [1] Select(description='Flux: tracked pixel, ..., name='Flux method', options=OrderedDict({'dynamic_spec...]), value='patch_fit', width=200)
        [2] LoadingSpinner(size=24)
    [1] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')
        [1] IPyWidget(HTML, height=150, sizing_mode='stretch_width')
    [2] Markdown(str, sizing_mode='stretch_width')
    [3] Bokeh(figure, height=420, sizing_mode='stretch_width')
    [4] IPyWidget(VBox, height=620, sizing_mode='stretch_width')

/home/claw/code/ovro-lwa-portal/src/ovro_lwa_portal/io.py:807: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  ds = xr.open_zarr(store, chunks=chunks, **kwargs)
/home/claw/code/ovro-lwa-portal/src/ovro_lwa_portal/io.py:807: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. 

## Parallelism notes

Jupiter QA Zarr stores use **per-time WCS** (`wcs_header_str(time)`), so RA/Dec
tracking maps a **different pixel each time step** before reading flux.

| Stage | `dynamic_spectrum` | `patch_fit` / `patch_max` |
| --- | --- | --- |
| **track** | Bulk header parse + in-process `world2pix` (one WCS object, CRVAL updated per time) | Same |
| **extract** | One vectorized Zarr read (`isel` with per-time `l`/`m` index arrays) — **threaded** dask | Same for point reads; patches still batched |
| **reduce / fit** | — | Per-time statistics or Gaussian fit — **process pool** locally, or distributed workers when a `Client` is active |

**Why one core before:** the default `dask.compute()` scheduler is threaded; scipy
Gaussian fitting holds the GIL, so `patch_fit` stayed on a single CPU. The
accessor now uses `scheduler="threads"` for I/O and `scheduler="processes"` for
CPU-bound patch work when no distributed `Client` is running.

**Recommendations**

- **`dynamic_spectrum`** — leave `USE_DASK_CLIENT = False`; threaded batched reads
  are usually fastest and avoid shipping tiny tasks to workers.
- **`patch_fit`** — default (no Client) is fine after the accessor change; set
  `USE_DASK_CLIENT = True` with `DASK_PROCESSES = True` only if you want the
  dashboard or to tune worker count / memory on very long days.
- Watch the activity log for phased progress (`Pixel track`, `Pixel I/O`, `Patch fit`).